# Medición: SVD float64 grande en GPU T4 frente a CPU de Kaggle (Hito B)

**Ejecutar en modo interactivo** (abrir el notebook y ejecutar celda a celda, **no** "Save Version"): así la salida
se ve en vivo. Accelerator = GPU T4 x2. Internet no hace falta. Sin descargas; ~5-25 min.
Copia las líneas `RESULTADO` y pásaselas a Claude. Si una celda tarda más de 20 min, interrúmpela y apúntalo.

In [ ]:
import os, time, torch, platform
print(torch.__version__, platform.processor(), "CPU hilos:", torch.get_num_threads(), "nproc:", os.cpu_count())
print(torch.cuda.get_device_name(0), torch.backends.mkl.is_available() if hasattr(torch.backends, "mkl") else None)
M, N = 18944, 8960   # covarianza de down_proj.in: Qwen2.5-7B × Qwen2.5-1.5B

In [ ]:
# 1) Acumulación float64 en GPU (lo que se hace por ejemplo y capa): [M,T]@[T,N] con T≈180
a = torch.randn(180, M, device="cuda", dtype=torch.float64); b = torch.randn(180, N, device="cuda", dtype=torch.float64)
C = torch.zeros(M, N, device="cuda", dtype=torch.float64)
torch.cuda.synchronize(); t = time.time()
for _ in range(10): C += a.T @ b
torch.cuda.synchronize(); dt = (time.time() - t) / 10
print(f"RESULTADO acumulacion_gpu_f64: {dt:.3f} s/ejemplo -> {dt*256:.0f} s por capa (256 ejemplos)")

In [ ]:
# 2) SVD float64 en CPU (MKL)
Cc = torch.randn(M, N, dtype=torch.float64)
t = time.time(); U, S, Vh = torch.linalg.svd(Cc, full_matrices=False); dt = time.time() - t
print(f"RESULTADO svd_cpu_f64: {dt:.0f} s por capa -> {dt*28/60:.0f} min para 28 capas")
del U, S, Vh

In [ ]:
# 3) SVD float64 en GPU T4 (puede ser muy lenta: interrumpe si pasa de 20 min y apúntalo)
Cg = torch.randn(M, N, device="cuda", dtype=torch.float64)
torch.cuda.synchronize(); t = time.time(); U, S, Vh = torch.linalg.svd(Cg, full_matrices=False); torch.cuda.synchronize()
dt = time.time() - t
print(f"RESULTADO svd_gpu_f64: {dt:.0f} s por capa -> {dt*28/60:.0f} min para 28 capas")

In [ ]:
# 4) Referencia: SVD float32 en GPU (más rápida, pero no válida por precisión; solo para comparar)
Cg32 = Cg.float(); del Cg, U, S, Vh; torch.cuda.empty_cache()
torch.cuda.synchronize(); t = time.time(); torch.linalg.svd(Cg32, full_matrices=False); torch.cuda.synchronize()
print(f"RESULTADO svd_gpu_f32: {time.time()-t:.0f} s")